This generates the "ready" csv files that are used to train the kmrf model
-- if new data is wanted, run "data_collection.ipynb" and "data_collection_us_traded.ipynb" first

In [20]:
import pandas as pd
import numpy as np
from urllib.request import urlopen
import certifi
import json
from fredapi import Fred
import os
import ssl

# Custom packages
import derive_data as dd

import warnings
warnings.filterwarnings("ignore")

# Environment variables
import dotenv
dotenv.load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
FMP_API_KEY = os.getenv("FMP_API_KEY")

In [21]:
fmp_idx = pd.read_csv('data/inputs/fmp_index_list.csv')
fmp_comm = pd.read_csv('data/inputs/fmp_commodity_list.csv')
us_equity_symbol_names = {
    # BOND ETFS
    'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    'IEF': 'iShares 7-10 Year Treasury Bond ETF',
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    '^RUI': 'Russell 1000',
    '^RUA': 'Russell 3000',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    'VOO': 'Vanguard S&P 500 ETF',
    'RSP': 'Invesco S&P 500 Equal Weight ETF',
    'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'QQQM': 'Invesco Nasdaq 100 ETF',
    'ONEQ': 'Fidelity Nasdaq Composite Index ETF',
    'IWM': 'iShares Russell 2000 ETF',
    'IWB': 'iShares Russell 1000 ETF',
    'IWV': 'iShares Russell 3000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS (SELECT SECTOR SPDRS)
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    'XLRE': 'Real Estate Select Sector SPDR',
    'XLC': 'Communication Services Select Sector SPDR',
    
    # GROWTH ETFs
    'IVW': 'iShares S&P 500 Growth ETF',
    'VONG': 'Vanguard Russell 1000 Growth ETF',
    'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    'VUG': 'Vanguard Growth ETF',
    'SPYG': 'SPDR Portfolio S&P 500 Growth ETF',
    
    # VALUE ETFs
    'IVE': 'iShares S&P 500 Value ETF',
    'VONV': 'Vanguard Russell 1000 Value ETF',
    'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    'VTV': 'Vanguard Value ETF',
    'SPYV': 'SPDR Portfolio S&P 500 Value ETF',
    
    # SIZE ETFs
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    'IJH': 'iShares Core S&P Mid-Cap ETF',
    'IJR': 'iShares Core S&P Small-Cap ETF',
    'MDY': 'SPDR S&P MidCap 400 ETF',
    'SLY': 'SPDR S&P 600 Small Cap ETF',
    'VO': 'Vanguard Mid-Cap ETF',
    'VB': 'Vanguard Small-Cap ETF',
    'SCHA': 'Schwab U.S. Small-Cap ETF',
    'SCHM': 'Schwab U.S. Mid-Cap ETF',
    'VTWO': 'Vanguard Russell 2000 ETF',
    'VTHR': 'Vanguard Russell 3000 ETF',
    'THRK': 'iShares Russell 3000 ETF',
    'SPSM': 'SPDR Portfolio S&P 600 Small Cap ETF',
    'SMLF': 'iShares Small-Cap US Equity Factor ETF',
    
    # NASDAQ SPECIFIC
    'QTEC': 'First Trust Nasdaq-100 Technology Sector Index Fund',
    'QQEW': 'First Trust Nasdaq-100 Equal Weighted Index Fund',
    'QQQG': 'Pacer Nasdaq 100 Top 50 Cash Cows Dividend Growth ETF',
    'QQQV': 'Pacer Nasdaq 100 Top 50 Value ETF',
    
    # DIVIDEND/QUALITY
    'SCHD': 'Schwab U.S. Dividend Equity ETF',
    'VYM': 'Vanguard High Dividend Yield ETF',
    'DVY': 'iShares Select Dividend ETF',
    'QUAL': 'iShares MSCI USA Quality Factor ETF',
    'USMV': 'iShares MSCI USA Min Vol Factor ETF',
    
    # EQUAL WEIGHT
    'EWSC': 'Invesco S&P SmallCap 600 Equal Weight ETF',
    'EWMC': 'Invesco S&P MidCap 400 Equal Weight ETF',
}
int_equity_symbol_names = {
    'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
}
macro_codes_dict = {
    # RATE BENCHMARKS
    'DFF': 'Federal Funds Effective Rate',
    'SOFR': 'Secured Overnight Financing Rate',
    # TREASURY RATES
    'DGS1MO': '1-Month Treasury Rate',
    'DGS3MO': '3-Month Treasury Rate', 
    'DGS6MO': '6-Month Treasury Rate',
    'DGS1': '1-Year Treasury Rate',
    'DGS2': '2-Year Treasury Rate',
    'DGS3': '3-Year Treasury Rate',
    'DGS5': '5-Year Treasury Rate',
    'DGS7': '7-Year Treasury Rate',
    'DGS10': '10-Year Treasury Rate',
    'DGS20': '20-Year Treasury Rate',
    'DGS30': '30-Year Treasury Rate',
    # OTHER RATES
    'DAAA': 'Moody\'s Seasoned AAA Corporate Bond Yield',
    'DBAA': 'Moody\'s Seasoned BAA Corporate Bond Yield',
    'OBMMCONF30YF': '30-Year Fixed Rate Conforming Mortgage Index',
    'DPRIME': 'Bank Prime Loan Rate',
    'T5YIE': '5-Year Breakeven Inflation Rate',
    'T10YIE': '10-Year Breakeven Inflation Rate',
    'T30YIE': '30-Year Breakeven Inflation Rate',
    # ECONOMIC INDICATORS
    'GDP': 'Gross Domestic Product',
    'GDPC1': 'Real Gross Domestic Product',
    'A939RX0Q048SBEA': 'Real GDP Per Capita',
    'PCE': 'Personal Consumption Expenditures',
    'PCEPI': 'Personal Consumption Expenditures Price Index',
    'PCEC96': 'Real Personal Consumption Expenditures',
    'CPIAUCSL': 'Consumer Price Index',
    'CPILFESL': 'Core CPI (Less Food and Energy)',
    'UNRATE': 'Unemployment Rate',
    'CIVPART': 'Labor Force Participation Rate',
    'INDPRO': 'Industrial Production Index',
    'PAYEMS': 'Total Nonfarm Payrolls',
    'HOUST': 'Housing Starts',
    'PERMIT': 'Building Permits',
    'MTSDS133FMS': 'Monthly US Government Surplus/Deficit',
    'GFDEGDQ188S': 'Federal Government Debt to GDP Ratio',
    'PMSAVE': 'Personal Savings',
    'PSAVERT': 'Personal Saving Rate',
    'GPDI': 'Gross Private Domestic Investment',
    'GPDIC1': 'Real Gross Private Domestic Investment',
    'BOGZ1FU263092001Q': 'Foreign Direct Investment in the United States',
    'QBPBSTAS': 'Balance Sheet - Total Assets',
    'FDHBFRBN': 'Federal Debt Held by Federal Reserve Banks',
    'FYGFDPUN': 'Federal Debt Held by the Public',
    'FDHBFIN': 'Federal Debt Held by Foreign Investors',
    'COMPOUT': 'Commercial Paper Outstanding',
    'ABCOMP': 'Asset-Backed Commercial Paper Outstanding',
    # MONEY SUPPLY
    'M1SL': 'M1 Money Stock',
    'M2SL': 'M2 Money Stock',
    'BASE': 'St. Louis Adjusted Monetary Base',
    # MARKET INDICATORS
    'VIXCLS': 'CBOE Volatility Index (VIX)',
    'UMCSENT': 'University of Michigan Consumer Sentiment',
    'USSLIND': 'Leading Index for the United States',
    'VISASMIHSA': 'Visa U.S. Consumer Spending Momentum Index: Headline',
    'VISASMIDSA': 'Visa U.S. Consumer Spending Momentum Index: Discretionary',
}
macro_units = {
    # INTEREST RATES AND FINANCIAL RATES
    'DFF': 'Percent, Seasonally Adjusted',
    'SOFR': 'Percent, Not Seasonally Adjusted',
    'DGS1MO': 'Percent, Not Seasonally Adjusted',
    'DGS3MO': 'Percent, Not Seasonally Adjusted',
    'DGS6MO': 'Percent, Not Seasonally Adjusted',
    'DGS1': 'Percent, Not Seasonally Adjusted',
    'DGS2': 'Percent, Not Seasonally Adjusted',
    'DGS3': 'Percent, Not Seasonally Adjusted',
    'DGS5': 'Percent, Not Seasonally Adjusted',
    'DGS7': 'Percent, Not Seasonally Adjusted',
    'DGS10': 'Percent, Not Seasonally Adjusted',
    'DGS20': 'Percent, Not Seasonally Adjusted',
    'DGS30': 'Percent, Not Seasonally Adjusted',
    'DAAA': 'Percent, Not Seasonally Adjusted',
    'DBAA': 'Percent, Not Seasonally Adjusted',
    'OBMMCONF30YF': 'Percent, Not Seasonally Adjusted',
    'DPRIME': 'Percent, Not Seasonally Adjusted',
    'T5YIE': 'Percent, Not Seasonally Adjusted',
    'T10YIE': 'Percent, Not Seasonally Adjusted',
    'T30YIE': 'Percent, Not Seasonally Adjusted',
    # ECONOMIC INDICATORS
    'GDP': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'GDPC1': 'Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate',
    'A939RX0Q048SBEA': 'Chained 2017 Dollars, Seasonally Adjusted',
    'PCE': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'PCEPI': 'Index 2017=100, Seasonally Adjusted',
    'PCEC96': 'Billions of Chained 2017 Dollars, Seasonally Adjusted',
    'CPIAUCSL': 'Index 1982-1984=100, Seasonally Adjusted',
    'CPILFESL': 'Index 1982-1984=100, Seasonally Adjusted',
    'UNRATE': 'Percent, Seasonally Adjusted',
    'CIVPART': 'Percent, Seasonally Adjusted',
    'INDPRO': 'Index 2017=100, Seasonally Adjusted',
    'PAYEMS': 'Thousands of Persons, Seasonally Adjusted',
    'HOUST': 'Thousands of Units, Seasonally Adjusted Annual Rate',
    'PERMIT': 'Thousands of Units, Seasonally Adjusted Annual Rate',
    'MTSDS133FMS': 'Millions of Dollars, Not Seasonally Adjusted',
    'GFDEGDQ188S': 'Percent of GDP, Not Seasonally Adjusted',
    'PMSAVE': 'Billions of Dollars, Not Seasonally Adjusted',
    'PSAVERT': 'Percent, Seasonally Adjusted',
    'GPDI': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'GPDIC1': 'Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate',
    'BOGZ1FU263092001Q': 'Millions of Dollars, Not Seasonally Adjusted',
    'QBPBSTAS': 'Millions of Dollars, Not Seasonally Adjusted',
    'FDHBFRBN': 'Billions of Dollars, Not Seasonally Adjusted',
    'FYGFDPUN': 'Millions of Dollars, Not Seasonally Adjusted',
    'FDHBFIN': 'Billions of Dollars, Not Seasonally Adjusted',
    'COMPOUT': 'Billions of Dollars, Not Seasonally Adjusted',
    'ABCOMP': 'Billions of Dollars, Not Seasonally Adjusted',
    # MONEY SUPPLY
    'M1SL': 'Billions of Dollars, Seasonally Adjusted',
    'M2SL': 'Billions of Dollars, Seasonally Adjusted',
    'BASE': 'Millions of Dollars, Not Seasonally Adjusted',
    # MARKET INDICATORS
    'VIXCLS': 'Index, Not Seasonally Adjusted',
    'UMCSENT': 'Index 1966:Q1=100, Not Seasonally Adjusted',
    'USSLIND': 'Percent, Seasonally Adjusted',
    'VISASMIHSA': 'Index, Seasonally Adjusted',
    'VISASMIDSA': 'Index, Seasonally Adjusted',
}


# US Equities

In [3]:
us_equity_data = pd.read_csv('data/processed/us_equity_all_data.csv', index_col=0, header=[0, 1], parse_dates=True)
us_equity_data.index = pd.to_datetime(us_equity_data.index)
us_equity_data.rename(columns=us_equity_symbol_names, level=0, inplace=True)

us_equity_always_disclude = ['Vanguard S&P 500 ETF', 'Real Estate Select Sector SPDR', 'Communication Services Select Sector SPDR']
us_equity_disclude = [name for name in us_equity_data.columns.get_level_values(0) if 'Russell 1000' in name or 'Russell 3000' in name]\
                        + ['S&P 500', 'Nasdaq Composite', 'Dow Jones Industrial Average', 'Nasdaq 100', 'Russell 2000']
us_treasuries = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']

us_equity_include = list(set(us_equity_data.columns.get_level_values(0).tolist()) - set(us_equity_always_disclude)\
                          - set(us_equity_disclude) - set(int_equity_symbol_names.keys()) - set(us_treasuries))
col_mask = us_equity_data.columns.map(lambda x: x[0] in us_equity_include)

us_equity_data = us_equity_data.loc[:, col_mask]
# us_equity_data.tail()

In [5]:
us_equity_assets = us_equity_data.columns.get_level_values(0).unique().tolist()
us_equity_derived = dd.TimeSeriesDerivedFields(price_data=us_equity_data.xs(us_equity_assets[0], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
us_equity_derived.columns = pd.MultiIndex.from_product([[us_equity_assets[0]], us_equity_derived.columns])
for i in range(1, len(us_equity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=us_equity_data.xs(us_equity_assets[i], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[us_equity_assets[i]], temp.columns])
    us_equity_derived = pd.concat([us_equity_derived, temp], axis=1)

us_equity_derived.to_csv('data/ready/us_equity.csv')
us_equity_derived.to_excel('data/ready/us_equity.xlsx')

Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 8202 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 16404/16404 [00:02<00:00, 7483.23it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5620 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 11240/11240 [00:01<00:00, 7819.95it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5513 out of 9002
Extracting tsfresh features...


...

Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 6063 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 12126/12126 [00:01<00:00, 7672.22it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5041 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 10082/10082 [00:01<00:00, 7077.08it/s]


Generated 20 tsfresh features


# US Treasury ETFs

In [6]:
us_equity_data = pd.read_csv('data/processed/us_equity_all_data.csv', index_col=0, header=[0, 1], parse_dates=True)
us_equity_data.index = pd.to_datetime(us_equity_data.index)
us_equity_data.rename(columns=us_equity_symbol_names, level=0, inplace=True)

us_treasuries = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']
col_mask = us_equity_data.columns.map(lambda x: x[0] in us_treasuries)

us_treasury_data = us_equity_data.loc[:, col_mask]
# us_treasury_data.tail()

In [7]:
us_treasury_assets = us_treasury_data.columns.get_level_values(0).unique().tolist()
us_treasury_derived = dd.TimeSeriesDerivedFields(price_data=us_treasury_data.xs(us_treasury_assets[0], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
us_treasury_derived.columns = pd.MultiIndex.from_product([[us_treasury_assets[0]], us_treasury_derived.columns])
for i in range(1, len(us_treasury_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=us_treasury_data.xs(us_treasury_assets[i], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[us_treasury_assets[i]], temp.columns])
    us_treasury_derived = pd.concat([us_treasury_derived, temp], axis=1)

us_treasury_derived.to_csv('data/ready/us_treasury.csv')
us_treasury_derived.to_excel('data/ready/us_treasury.xlsx')

Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 4593 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 9186/9186 [00:01<00:00, 8084.78it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5811 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 11622/11622 [00:01<00:00, 7341.40it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5811 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 11622/11622 [00:01<00:00, 8273.23it/s]


Generated 20 tsfresh features


# International Equities

In [8]:
int_equity_data = pd.read_csv('data/processed/us_equity_all_data.csv', index_col=0, header=[0, 1], parse_dates=True)
int_equity_data.index = pd.to_datetime(int_equity_data.index)
int_equity_data.rename(columns=int_equity_symbol_names, level=0, inplace=True)

col_mask = int_equity_data.columns.map(lambda x: x[0] in int_equity_symbol_names.values())

int_equity_data = int_equity_data.loc[:, col_mask]
# int_equity_data.tail()

In [9]:
int_equity_assets = int_equity_data.columns.get_level_values(0).unique().tolist()
int_equity_derived = dd.TimeSeriesDerivedFields(price_data=int_equity_data.xs(int_equity_assets[0], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
int_equity_derived.columns = pd.MultiIndex.from_product([[int_equity_assets[0]], int_equity_derived.columns])
for i in range(1, len(int_equity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=int_equity_data.xs(int_equity_assets[i], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[int_equity_assets[i]], temp.columns])
    int_equity_derived = pd.concat([int_equity_derived, temp], axis=1)

int_equity_derived.to_csv('data/ready/int_equity.csv')
int_equity_derived.to_excel('data/ready/int_equity.xlsx')

Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 3668 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 7336/7336 [00:00<00:00, 7775.56it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 4553 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 9106/9106 [00:01<00:00, 8133.72it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5151 out of 9002
Extracting tsfresh features...


...

Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 7411 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 14822/14822 [00:01<00:00, 8185.41it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 3412 out of 9002
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 6824/6824 [00:00<00:00, 6992.75it/s]


Generated 20 tsfresh features


# Commodities

In [10]:
comm_symbol_name_dict = fmp_comm.set_index('symbol')['name'].to_dict()
comm_symbol_name_dict.update({'Nickel': 'Nickel'})

commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0,1])
commodity_data.index = pd.to_datetime(commodity_data.index)

commodity_data.rename(columns=comm_symbol_name_dict, level=0, inplace=True)

# remove volume and vwap columns
commodity_data = commodity_data.loc[:, commodity_data.columns.map(lambda x: x[1] not in ['volume', 'vwap'])]
# commodity_data.tail()

In [11]:
commodity_assets = commodity_data.columns.get_level_values(0).unique().tolist()
commodity_equity_derived = dd.TimeSeriesDerivedFields(price_data=commodity_data.xs(commodity_assets[0], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=False)
commodity_equity_derived.columns = pd.MultiIndex.from_product([[commodity_assets[0]], commodity_equity_derived.columns])
for i in range(1, len(commodity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=commodity_data.xs(commodity_assets[i], level=0, axis=1)).compute_all_derived_fields(include_tsfresh=False)
    temp.columns = pd.MultiIndex.from_product([[commodity_assets[i]], temp.columns])
    commodity_equity_derived = pd.concat([commodity_equity_derived, temp], axis=1)
    
commodity_equity_derived.to_csv('data/ready/commodity.csv')
commodity_equity_derived.to_excel('data/ready/commodity.xlsx')

# Macro Data

In [32]:
macro_data_daily = pd.read_csv('data/processed/macro_data_daily.csv', parse_dates=['date'], index_col='date')
# macro_data_monthly = pd.read_csv('data/processed/macro_data_monthly.csv', parse_dates=['date'], index_col='date')
# macro_data_quarterly = pd.read_csv('data/processed/macro_data_quarterly.csv', parse_dates=['date'], index_col='date')

macro_data_daily.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}') for col in macro_data_daily.columns])
# macro_data_monthly.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}, monthly') for col in macro_data_monthly.columns])
# macro_data_quarterly.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}, quarterly') for col in macro_data_quarterly.columns])

## Forward fill less frequently updated macro data
# macro_data_monthly_ffill = macro_data_monthly.reindex(macro_data_daily.index).ffill()
# macro_data_quarterly_ffill = macro_data_quarterly.reindex(macro_data_daily.index).ffill()

# macro_data_all = pd.concat([macro_data_daily, macro_data_monthly_ffill, macro_data_quarterly_ffill], axis=1)

# macro_data_all.to_csv('data/ready/macro_data_all.csv')
# macro_data_all.to_excel('data/ready/macro_data_all.xlsx')

In [ ]:
macro_data_daily_lagged = macro_data_daily.shift(1)
macro_data_daily_lagged.columns = pd.MultiIndex.from_tuples([(f'{col[0]} lag1d', f'{col[1]}') for col in macro_data_daily_lagged.columns])


In [ ]:
# macro_data_daily_lagged.iloc[1:].to_csv('data/ready/macro_data_all.csv')
# macro_data_daily_lagged.iloc[1:].to_excel('data/ready/macro_data_all.xlsx')

# 